# JetClass Preparation Notebook

This notebook turns the preparation checklist into a runnable JetClass data-quality workflow.

It is intentionally scoped to the preparation stage:
- discover JetClass HDF5 files
- inspect file structure, sample shapes, masks, and labels
- build a reproducible file-level split manifest
- define the A-D feature configurations used by the research plan
- provide helpers that downstream training notebooks can reuse


## Notebook Scope

This notebook covers the preparation layer of the JetClass workflow.

It is responsible for:
- discovering JetClass HDF5 files
- inspecting file structure, sample shapes, masks, and labels
- freezing a reproducible file-level split manifest
- defining the A-D feature ladder used by the research plan
- providing experiment-record helpers for downstream training notebooks


In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np

try:
    import h5py
    H5PY_AVAILABLE = True
except ImportError:
    h5py = None
    H5PY_AVAILABLE = False
    print("h5py is not installed in this kernel. JetClass inspection cells will be unavailable until h5py is installed.")

np.set_printoptions(suppress=True, precision=4)

SEED = 42
RNG = np.random.default_rng(SEED)

def find_repo_root(start: Path) -> Path:
    """Walk upward until the repository root is found."""
    for candidate in (start, *start.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "docs").exists():
            return candidate
    return start

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
DATA_ROOT_ENV = os.getenv("JETCLASS_DATA_ROOT")
DATA_ROOT = Path(DATA_ROOT_ENV).expanduser().resolve() if DATA_ROOT_ENV else None

DEFAULT_DATA_DIRS = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "src" / "data",
    PROJECT_ROOT / "artifacts" / "jetclass",
]

SPLIT_MANIFEST_PATH = PROJECT_ROOT / "data" / "split_manifest.json"

PARTICLE_DATASET_NAMES = ("data", "particles", "particle_data")
JET_DATASET_NAMES = ("jet", "jets", "jet_features")
LABEL_DATASET_NAMES = ("pid", "label", "labels")
MASK_DATASET_NAMES = ("mask", "masks", "particle_mask", "particle_masks")

# Mask is tracked separately from the feature tensor.
FEATURE_CONFIGS = {
    "A": ("rel_eta", "rel_phi", "log_pt"),
    "B": ("rel_eta", "rel_phi", "log_pt", "pid_compact"),
    "C": ("rel_eta", "rel_phi", "log_pt", "pid_compact", "charge"),
    "D": (
        "rel_eta",
        "rel_phi",
        "log_pt",
        "pid_compact",
        "charge",
        "energy_fraction",
        "momentum_fraction",
    ),
}

print(f"Project root: {PROJECT_ROOT}")
print(f"JETCLASS_DATA_ROOT: {DATA_ROOT if DATA_ROOT else 'not set'}")
print(f"Split manifest path: {SPLIT_MANIFEST_PATH}")
print(f"h5py available: {H5PY_AVAILABLE}")


In [ ]:
def _unique_paths(paths: Iterable[Path]) -> list[Path]:
    seen: set[Path] = set()
    ordered: list[Path] = []
    for raw_path in paths:
        path = Path(raw_path).expanduser().resolve()
        if path not in seen:
            seen.add(path)
            ordered.append(path)
    return ordered


def discover_hdf5_files(roots: Sequence[Path] | None = None) -> list[Path]:
    """Discover HDF5 files under the configured search roots."""
    search_roots: list[Path] = []
    if DATA_ROOT is not None:
        search_roots.append(DATA_ROOT)
    if roots is not None:
        search_roots.extend(Path(root) for root in roots)
    else:
        search_roots.extend(DEFAULT_DATA_DIRS)

    candidates: list[Path] = []
    for root in search_roots:
        root = Path(root).expanduser().resolve()
        if not root.exists():
            continue
        if root.is_file() and root.suffix.lower() in {'.h5', '.hdf5', '.hdf'}:
            candidates.append(root)
            continue
        if root.is_dir():
            for file_path in root.rglob('*'):
                if file_path.suffix.lower() in {'.h5', '.hdf5', '.hdf'}:
                    candidates.append(file_path)

    return _unique_paths(candidates)


def relpath(path: Path) -> str:
    path = Path(path).expanduser().resolve()
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def first_existing_key(mapping, candidates: Sequence[str]) -> str | None:
    for key in candidates:
        if key in mapping:
            return key
    return None


def get_dataset(mapping, candidates: Sequence[str]):
    key = first_existing_key(mapping, candidates)
    return mapping[key] if key is not None else None


In [ ]:
def dataset_info(obj) -> dict[str, object]:
    if not H5PY_AVAILABLE:
        raise ImportError("h5py is required to inspect JetClass HDF5 files.")
    if isinstance(obj, h5py.Dataset):
        return {
            "kind": "dataset",
            "shape": None if obj.shape is None else tuple(obj.shape),
            "dtype": str(obj.dtype),
            "compression": obj.compression,
            "chunks": obj.chunks,
        }
    if isinstance(obj, h5py.Group):
        return {
            "kind": "group",
            "keys": list(obj.keys()),
        }
    return {"kind": type(obj).__name__}


def print_hdf5_summary(file_path: Path) -> None:
    if not H5PY_AVAILABLE:
        raise ImportError("h5py is required to inspect JetClass HDF5 files.")
    with h5py.File(file_path, "r") as f:
        print()
        print(f"File: {relpath(file_path)}")
        for key in f.keys():
            print(f"  {key}: {dataset_info(f[key])}")


def to_python_scalar(value):
    if isinstance(value, np.generic):
        return value.item()
    return value


def infer_mask(particles: np.ndarray, provided_mask: np.ndarray | None = None, eps: float = 0.0) -> np.ndarray:
    """Infer a particle-validity mask when the file does not ship one explicitly."""
    if provided_mask is not None:
        return np.asarray(provided_mask).astype(bool)
    particles = np.asarray(particles)
    if particles.ndim < 2:
        raise ValueError(f"Expected at least 2 dimensions for particles, got {particles.shape}")
    return np.any(np.isfinite(particles) & (np.abs(particles) > eps), axis=-1)


def summarize_labels(labels: np.ndarray, limit: int | None = None) -> list[dict[str, object]]:
    flat = np.asarray(labels).reshape(-1)
    if limit is not None:
        flat = flat[:limit]
    values, counts = np.unique(flat, return_counts=True)
    return [
        {"label": to_python_scalar(value), "count": int(count)}
        for value, count in zip(values, counts, strict=False)
    ]


def particle_counts_from_mask(mask: np.ndarray):
    mask = np.asarray(mask).astype(bool)
    if mask.ndim == 1:
        return int(mask.sum())
    if mask.ndim < 2:
        raise ValueError(f"Expected at least 1 dimension for mask, got {mask.shape}")
    return mask.sum(axis=-1)


In [ ]:
def inspect_sample(file_path: Path, sample_idx: int = 0) -> dict[str, np.ndarray | None]:
    """Load one jet and print the shapes, mask information, and label."""
    if not H5PY_AVAILABLE:
        raise ImportError("h5py is required to inspect JetClass HDF5 files.")
    with h5py.File(file_path, "r") as f:
        particle_ds = get_dataset(f, PARTICLE_DATASET_NAMES)
        jet_ds = get_dataset(f, JET_DATASET_NAMES)
        label_ds = get_dataset(f, LABEL_DATASET_NAMES)
        mask_ds = get_dataset(f, MASK_DATASET_NAMES)

        if particle_ds is None:
            raise KeyError(f"No particle dataset found in {relpath(file_path)}")

        particles = np.asarray(particle_ds[sample_idx])
        jet = None if jet_ds is None else np.asarray(jet_ds[sample_idx])
        label = None if label_ds is None else np.asarray(label_ds[sample_idx])
        mask = None if mask_ds is None else np.asarray(mask_ds[sample_idx])
        inferred_mask = infer_mask(particles, mask)
        valid_particle_count = int(np.asarray(inferred_mask).sum())

        print(f"File: {relpath(file_path)}")
        print(f"Particle tensor shape: {particles.shape}, dtype: {particles.dtype}")
        if jet is not None:
            print(f"Jet tensor shape: {jet.shape}, dtype: {jet.dtype}")
        if label is not None:
            flat_label = np.asarray(label).reshape(-1)
            print(f"Label: {to_python_scalar(flat_label[0]) if flat_label.size == 1 else flat_label.tolist()}")
        print(f"Valid particles in sample {sample_idx}: {valid_particle_count}")
        print("First particles:")
        print(particles[:5])

        return {
            "particles": particles,
            "jet": jet,
            "label": label,
            "mask": mask,
            "inferred_mask": inferred_mask,
        }


In [ ]:
hdf5_files = discover_hdf5_files()
print(f'Found {len(hdf5_files)} HDF5 file(s).')

if not hdf5_files:
    print('No JetClass files were found. Set JETCLASS_DATA_ROOT or place the dataset under one of the default search directories.')
else:
    for file_path in hdf5_files[:3]:
        print_hdf5_summary(file_path)

    sample_file = hdf5_files[0]
    _sample = inspect_sample(sample_file, sample_idx=0)


## Fixed Split Manifest

The research plan requires immutable train/validation/test splits.

This notebook uses file-level splitting as the default, and it prefers existing `train` / `val` / `test` naming when the dataset already comes pre-partitioned.

The manifest is stored as JSON so every later run can reuse the exact same file assignment.


In [ ]:
def infer_named_splits(file_paths: Sequence[Path]) -> dict[str, list[Path]] | None:
    splits = {'train': [], 'val': [], 'test': []}
    for file_path in file_paths:
        name = Path(file_path).name.lower()
        if 'train' in name:
            splits['train'].append(Path(file_path))
        elif 'val' in name or 'valid' in name:
            splits['val'].append(Path(file_path))
        elif 'test' in name:
            splits['test'].append(Path(file_path))
    if any(splits.values()):
        return splits
    return None


def build_file_split_manifest(
    file_paths: Sequence[Path],
    seed: int = SEED,
    ratios: tuple[float, float, float] = (0.8, 0.1, 0.1),
) -> dict[str, object]:
    files = _unique_paths(file_paths)
    if not files:
        raise ValueError('No files were provided for split generation.')
    if not np.isclose(sum(ratios), 1.0):
        raise ValueError(f'Split ratios must sum to 1.0, got {ratios!r}')

    named = infer_named_splits(files)
    if named is not None:
        splits = named
    else:
        shuffled = files.copy()
        rng = np.random.default_rng(seed)
        rng.shuffle(shuffled)
        n_total = len(shuffled)
        n_train = int(round(n_total * ratios[0]))
        n_val = int(round(n_total * ratios[1]))
        n_train = min(n_train, n_total)
        n_val = min(n_val, max(0, n_total - n_train))
        n_test = max(0, n_total - n_train - n_val)
        splits = {
            'train': shuffled[:n_train],
            'val': shuffled[n_train:n_train + n_val],
            'test': shuffled[n_train + n_val:n_train + n_val + n_test],
        }

    manifest = {
        'seed': seed,
        'ratios': list(ratios),
        'created_at': datetime.now(timezone.utc).isoformat(),
        'project_root': str(PROJECT_ROOT),
        'splits': {
            split_name: [relpath(path) for path in split_files]
            for split_name, split_files in splits.items()
        },
    }
    return manifest


def save_split_manifest(manifest: dict[str, object], path: Path = SPLIT_MANIFEST_PATH) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
    return path


def load_split_manifest(path: Path = SPLIT_MANIFEST_PATH) -> dict[str, object]:
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


In [ ]:
if hdf5_files:
    split_manifest = build_file_split_manifest(hdf5_files, seed=SEED)
    saved_path = save_split_manifest(split_manifest)
    print(f'Split manifest saved to: {saved_path}')
    print(json.dumps(split_manifest, indent=2, ensure_ascii=False))
else:
    print('Skipping split manifest creation because no HDF5 files were discovered.')


## A-D Feature Configurations

The feature ladder is now aligned with the research plan:

- **A**: relative eta, relative phi, log transverse momentum, and mask
- **B**: A plus compact particle identification encoding
- **C**: B plus electric charge
- **D**: C plus energy and momentum fraction features

The exact raw column order still has to be confirmed from the HDF5 files. That is intentional: the notebook now separates the *semantic* configuration from the *file-specific* column mapping so the preprocessing code stays reusable.


In [ ]:
def validate_column_map(column_map: dict[str, int], config_name: str) -> list[int]:
    if config_name not in FEATURE_CONFIGS:
        raise KeyError(f'Unknown feature config: {config_name!r}')
    required_names = FEATURE_CONFIGS[config_name]
    missing = [name for name in required_names if name not in column_map]
    if missing:
        raise KeyError(f'Missing columns for config {config_name}: {missing}')
    return [column_map[name] for name in required_names]


def extract_feature_tensor(
    raw_particles: np.ndarray,
    column_map: dict[str, int],
    config_name: str,
) -> np.ndarray:
    indices = validate_column_map(column_map, config_name)
    raw_particles = np.asarray(raw_particles)
    return np.take(raw_particles, indices, axis=-1)


def remap_labels(labels: np.ndarray, label_map: dict[int, int]) -> np.ndarray:
    flat = np.asarray(labels).reshape(-1)
    unknown = sorted({to_python_scalar(value) for value in np.unique(flat)} - set(label_map.keys()))
    if unknown:
        raise KeyError(f'Label map is missing values: {unknown}')
    remapped = np.vectorize(lambda value: label_map[to_python_scalar(value)])(flat)
    return remapped.astype(np.int64)


### 9.2.3 Run a Simple Baseline

The actual training loop lives in `training/`, but this notebook keeps the experiment contract explicit.

Minimum expectation for the baseline stage:
- define the task as binary top-tagging
- reuse the frozen file-level split manifest
- record the input configuration, seed, and metric set
- compare at least one simple baseline against the PET / OmniLearn reference setup

### 9.2.4 Compare Against OmniLearn / PET-Style Reference Implementations

The preparation notebook does not host the full model code, but it should define the comparison frame:
- same JetClass source files
- same split manifest
- same feature configuration A-D
- same evaluation metrics and seed policy

### 9.2.5 Log Seed, Configuration, and Metrics for Each Experiment

Every run should emit a structured record so that results are reproducible and easy to compare across configurations.


In [ ]:
EXPERIMENT_LOG_DIR = PROJECT_ROOT / "experiments"


def build_experiment_record(
    name: str,
    seed: int,
    config_name: str,
    metrics: dict[str, float | int | None],
    split_manifest_path: Path | None = None,
    notes: str | None = None,
) -> dict[str, object]:
    """Create a structured experiment record for preparation-stage runs."""
    return {
        "name": name,
        "seed": int(seed),
        "config_name": config_name,
        "metrics": metrics,
        "split_manifest_path": None if split_manifest_path is None else relpath(split_manifest_path),
        "notes": notes,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


def save_experiment_record(record: dict[str, object], out_dir: Path = EXPERIMENT_LOG_DIR) -> Path:
    """Write a JSON experiment record that can be reused by training notebooks."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    timestamp = str(record.get("timestamp_utc", datetime.now(timezone.utc).isoformat())).replace(":", "-")
    safe_name = str(record["name"]).strip().lower().replace(" ", "-")
    out_path = out_dir / f"{timestamp}_{safe_name}.json"
    with out_path.open("w", encoding="utf-8") as handle:
        json.dump(record, handle, indent=2, ensure_ascii=False)
    return out_path


def summarize_metric_bundle(metrics: dict[str, float | int | None]) -> str:
    """Return a compact human-readable summary for notebook output."""
    parts = []
    for key in ("accuracy", "auc", "background_rejection"):
        if key in metrics and metrics[key] is not None:
            parts.append(f"{key}={metrics[key]}")
    return ", ".join(parts) if parts else "no metrics recorded"


## Passing Criteria

The preparation phase is ready when:

- JetClass files can be discovered and opened reliably
- one sample can be inspected with particle, jet, mask, and label information
- a split manifest can be generated and reused
- the A-D feature ladder is represented consistently in code
- the notebook fails loudly when the dataset path is missing instead of silently pretending to work
